# Bronze Layer — Raw Data Ingestion (Lakeflow Spark Declarative Pipeline)

User documentation for **`bronze_ingestion.py`** (this file's folder) — the Spark Declarative Pipeline (SDP) refactor of the legacy `Applebees_Pipeline_bronzelayer` notebook.

> **Note:** This notebook is documentation only and lives next to the pipeline source file for discoverability. It is *not* pipeline source code: the pipeline's libraries reference only `bronze_ingestion.py` (an explicit file entry, not a folder), so this notebook is **never executed** as part of a pipeline run. The code cells at the bottom are runnable verification queries you can execute manually to inspect the materialized bronze tables.

## What the bronze layer does

Ingests the 5 raw Applebee's JSON sources **"as-is"** (no joins, filters, deduplication, or transformations are applied) from the Unity Catalog volume `/Volumes/workspace/bronze/rawlanding/` using **Auto Loader** for incremental, exactly-once ingestion, then explodes the nested arrays of the `checks` fact table into their own bronze tables.

**Key steps**

1. **Raw ingestion** — 5 Auto Loader streaming tables, one per raw source folder
2. **Auditability** — every table carries `_ingested_at` (timestamp when the row was processed, `current_timestamp()`) and `_source_file` (path of the source file the row came from, `_metadata.file_path`)
3. **Array explosion** — the `Discounts`, `ItemsSold`, `Payments`, `Tax`, and `Voids` arrays of `checks` are exploded into 5 child tables with the exact same column names and order as the legacy notebook
4. **Verification** — runnable row-count, uniqueness, and coverage checks at the bottom of this notebook

## Where the tables live

Pipeline **`beestats_medallion_pipeline`** publishes all datasets to **`workspace.bronze`** (catalog `workspace`, schema `bronze`).

## Dataset inventory — 10 datasets in `workspace.bronze`

| Dataset | Source (rawlanding volume) | Ingestion | Notes |
| --- | --- | --- | --- |
| `checks` | `checks/dt=YYYY-MM-DD/checks.json` | Auto Loader, **text** format + `from_json` | 95 daily files, 71–168 MB each (single JSON array per file) — see next section |
| `employees` | `employees/` | Auto Loader, json format | Schema pinned from the legacy table (landing folder currently empty) |
| `items_table` | `items/` | Auto Loader, json format | Schema pinned from the legacy table (landing folder currently empty) |
| `locations_with_coordinates` | `locations/` | Auto Loader, json format | Schema pinned from the legacy table (landing folder currently empty) |
| `revenue_centers` | `revenue_centers/` | Auto Loader, json format | Schema pinned from the legacy table (landing folder currently empty) |
| `checks_discounts` | derived: `checks.Discounts` | streaming read of `checks` + `explode` | struct fields flattened to top-level columns |
| `checks_items` | derived: `checks.ItemsSold` | streaming read of `checks` + `explode` | keeps the nested `Discounts` array as-is |
| `checks_payments` | derived: `checks.Payments` | streaming read of `checks` + `explode` | |
| `checks_tax` | derived: `checks.Tax` | streaming read of `checks` + `explode` | |
| `checks_voids` | derived: `checks.Voids` | streaming read of `checks` + `explode` | |

Table and column names are identical to the legacy `workspace.bronze_schema` tables, including the same audit metadata columns. The exploded child tables keep `_source_file` lineage to the original raw file and add a fresh `_ingested_at`.

Note that folder names do not always match table names: `items/` → `items_table`, `locations/` → `locations_with_coordinates` (same mapping as the legacy notebook).

## Why `checks` uses the text format + `from_json`

Each daily `checks.json` file is a **single JSON array** of check records, 71–168 MB per file. Auto Loader's JSON reader parses such a file with the `multiLine` whole-file mode, which on **serverless compute** exceeds the per-task memory budget: Photon's simdjson reader reserves ~768 MiB for a large file against a ~1.1 GiB task budget, so files above ~85 MB fail with `SparkOutOfMemoryError`.

This workspace runs on Databricks Free Edition (serverless-only), so neither Photon nor the memory settings can be changed. The pipeline therefore reads each file as **one text line** (`cloudFiles.format = "text"`) and parses it with the JVM `from_json` parser, which has a much smaller memory footprint:

* `cloudFiles.maxFilesPerTrigger = 5` — small, memory-safe micro-batches
* `spark.sql.files.maxPartitionBytes = "16MB"` (per-flow `spark_conf`) — one whole file per scan task (each file is a single line and cannot be split)
* Pinned reader schema `value STRING, dt DATE` plus `cloudFiles.partitionColumns = "dt"` — required because Auto Loader would otherwise reuse the schema persisted by the flow's earlier JSON-format runs, and the `dt=YYYY-MM-DD` partition column must be declared in the provided schema
* `from_json(value, ARRAY<STRUCT<...>>)` with a DDL schema (`_CHECKS_SCHEMA` in the source file) that matches the legacy `bronze_schema.checks` table **1:1** — every nested array included — followed by `explode` to one row per check record

The 4 dimension tables keep the plain JSON reader — their folders are currently empty and their files are expected to be small. **If a future dimension file exceeds ~85 MB, apply the same text + `from_json` pattern to that table.**

## Empty dimension folders

`employees/`, `items/`, `locations/`, and `revenue_centers/` in the landing volume are currently **empty**. Auto Loader cannot infer a schema from zero files (`CF_EMPTY_DIR_FOR_SCHEMA_INFERENCE`), so each dimension table's schema is pinned 1:1 from the legacy bronze tables. The tables exist today with 0 rows and will ingest incrementally as soon as files land in those folders — just run a pipeline update.

## Running the pipeline

* **Dry run** — validates all 10 dataset definitions without processing data; the fast check to run after any code change
* **Update** — runs a regular incremental update: Auto Loader ingests any new files in the landing volume and the exploded child tables process the new `checks` rows. Re-running an update after new files land is all that is needed — no full refresh required
* **Pipeline source file:** `bronze_ingestion.py` (this folder) — the only file in the pipeline libraries. Notebooks (including this one) must **not** be added to the libraries; a folder/glob library entry would pick them up and fail

In [0]:
# ============================================================================
# BRONZE VERIFICATION — row counts for all 10 bronze tables
# ============================================================================
bronze_tables_all = [
    "checks",
    "checks_discounts",
    "checks_items",
    "checks_payments",
    "checks_tax",
    "checks_voids",
    "employees",
    "items_table",
    "locations_with_coordinates",
    "revenue_centers",
]

print("=" * 72)
print("  Bronze layer deliverables — workspace.bronze")
print("=" * 72)
for tbl in bronze_tables_all:
    full_name = f"workspace.bronze.{tbl}"
    try:
        print(f"  {tbl:<35} {spark.table(full_name).count():>12,} rows")
    except Exception:
        print(f"  {tbl:<35} {'(not found)':>12}")
print("=" * 72)

display(spark.sql("SHOW TABLES IN workspace.bronze"))

In [0]:
# ============================================================================
# CHECKS INTEGRITY — completeness and uniqueness
# ============================================================================
# Total rows, distinct source files (Auto Loader ingests each file exactly
# once), distinct check keys, and the covered date range.
display(spark.sql("""
    SELECT
        count(*)                                     AS total_rows,
        count(DISTINCT _source_file)                 AS distinct_files,
        count(DISTINCT (Id, LocationId, BusinessDate)) AS distinct_check_keys,
        min(dt)                                      AS first_day,
        max(dt)                                      AS last_day
    FROM workspace.bronze.checks
"""))

# Duplicate check keys per day (should always be 0).
display(spark.sql("""
    SELECT count(*) AS duplicate_keys FROM (
        SELECT Id, LocationId, BusinessDate, dt
        FROM workspace.bronze.checks
        GROUP BY Id, LocationId, BusinessDate, dt
        HAVING count(*) > 1
    )
"""))

In [0]:
# ============================================================================
# OPTIONAL — per-day comparison with the legacy batch tables
# ============================================================================
# Compares workspace.bronze.checks against the legacy
# workspace.bronze_schema.checks (loaded by the old notebook). On every
# shared day the counts should match exactly. "legacy only" days are not
# present in the rawlanding volume; "MISMATCH" indicates a real problem.
from pyspark.sql.functions import col, lit, when

new_per_dt = spark.sql("SELECT dt, count(*) AS n FROM workspace.bronze.checks GROUP BY dt")
legacy_per_dt = spark.sql("SELECT dt, count(*) AS n FROM workspace.bronze_schema.checks GROUP BY dt")

display(
    new_per_dt.alias("new").join(legacy_per_dt.alias("old"), ["dt"], "full_outer")
    .select(
        col("dt"),
        col("old.n").alias("legacy_rows"),
        col("new.n").alias("bronze_rows"),
        when(col("old.n").isNull(), lit("rawlanding only"))
        .when(col("new.n").isNull(), lit("legacy only"))
        .when(col("old.n") == col("new.n"), lit("match"))
        .otherwise(lit("MISMATCH"))
        .alias("per_day_status"),
    )
    .orderBy("dt")
)

In [0]:
# ============================================================================
# BRONZE TABLE PREVIEWS — 10-row samples + the checks schema
# ============================================================================
for tbl in ["checks", "checks_discounts", "checks_items", "checks_payments", "checks_tax", "checks_voids"]:
    print(f"▶ workspace.bronze.{tbl} (first 10 rows)")
    display(spark.table(f"workspace.bronze.{tbl}").limit(10))
    print("=" * 70)

# Schema of the core checks fact table as materialized by the pipeline.
spark.table("workspace.bronze.checks").printSchema()

## Known data-coverage notes (as of 2026-11-18)

* `rawlanding/checks` contains **95 days**: `dt=2026-04-28` → `dt=2026-07-31`, ingested from 95 files with 1,799,530 check rows and **zero duplicate keys**
* The legacy `workspace.bronze_schema.checks` covers **114 days** through `2026-08-20`; the **20 days of 2026-08-01 → 2026-08-20 are not present** in the landing volume
* `dt=2026-06-20` (20,084 rows) exists in the landing volume but was missing from the legacy tables
* On every one of the 94 shared days, `workspace.bronze.checks` row counts matched the legacy table **exactly** — no records are lost or duplicated by the text + `from_json` parsing
* The 4 dimension tables have **0 rows** because their landing folders are empty (schemas pinned, ready to ingest when files land)

## Operational constraints of this workspace

* Databricks Free Edition: **serverless-only** (no classic compute), Photon always on (cannot be disabled), and `spark.memory.fraction` is rejected for serverless pipelines
* Any Auto Loader flow whose files are large single JSON arrays should use the text + `from_json` pattern documented above rather than `multiLine` JSON
* When changing an Auto Loader flow's format mid-flight (e.g. json → text), pin the reader schema explicitly — otherwise Auto Loader reuses the schema persisted from earlier runs of that same flow